# 1 · From files to a table

`id, gender, age, interest` — and interests keep changing.

In [20]:
#@title Setup — run before the session
import os, shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.3', 'delta-spark==3.3.2'], check=True)
if not shutil.which('java'):
    subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless', shell=True, check=True)

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder.master('local[2]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.databricks.delta.properties.defaults.enableDeletionVectors', 'false')
    .config('spark.ui.showConsoleProgress', 'false'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

PATH = '/tmp/delta_talk/profiles'
shutil.rmtree('/tmp/delta_talk', ignore_errors=True)
print('Spark', spark.version, 'ready')

Spark 3.5.3 ready


In [21]:
#@title Helpers
def ls(path):
    print(sorted(f for f in os.listdir(path) if not f.startswith('.')))

def files_on_disk():
    return sorted(f for f in os.listdir(PATH) if f.endswith('.parquet'))

def files_in_table():
    return sorted(os.path.basename(f) for f in spark.read.format('delta').load(PATH).inputFiles())

def commit(version):
    return spark.read.json(f'{PATH}/_delta_log/{version:020d}.json')

## Create a Delta table

In [22]:
profiles = spark.createDataFrame(
    [(1, 'Male', 25, 'Sports'),
     (2, 'Female', 30, 'News'),
     (3, 'Male', 35, 'Social'),
     (4, 'Female', 28, 'Fashion')],
    'id INT, gender STRING, age INT, interest STRING')

profiles.coalesce(1).write.format('delta').save(PATH)

In [23]:
spark.read.format('delta').load(PATH).show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|  Sports|
|  2|Female| 30|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



## What did that write to disk?

In [24]:
ls(PATH)
ls(PATH + '/_delta_log')

['_delta_log', 'part-00000-4994dd5e-bec4-4ce9-b4fa-fe196ab77bd2-c000.snappy.parquet']
['00000000000000000000.crc', '00000000000000000000.json', '_commits']


## Open version 0

In [26]:
v0 = commit(0)
v0.printSchema()

root
 |-- add: struct (nullable = true)
 |    |-- dataChange: boolean (nullable = true)
 |    |-- modificationTime: long (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- size: long (nullable = true)
 |    |-- stats: string (nullable = true)
 |-- commitInfo: struct (nullable = true)
 |    |-- engineInfo: string (nullable = true)
 |    |-- isBlindAppend: boolean (nullable = true)
 |    |-- isolationLevel: string (nullable = true)
 |    |-- operation: string (nullable = true)
 |    |-- operationMetrics: struct (nullable = true)
 |    |    |-- numFiles: string (nullable = true)
 |    |    |-- numOutputBytes: string (nullable = true)
 |    |    |-- numOutputRows: string (nullable = true)
 |    |-- operationParameters: struct (nullable = true)
 |    |    |-- mode: string (nullable = true)
 |    |    |-- partitionBy: string (nullable = true)
 |    |-- timestamp: long (nullable = true)
 |    |-- txnId: string (nullable = true)
 |-- metaData: struct (nullable = true)
 |    |

In [27]:
add = v0.where('add is not null').first().add
print(add.path)
print(add.stats)

part-00000-4994dd5e-bec4-4ce9-b4fa-fe196ab77bd2-c000.snappy.parquet
{"numRecords":4,"minValues":{"id":1,"gender":"Female","age":25,"interest":"Fashion"},"maxValues":{"id":4,"gender":"Male","age":35,"interest":"Sports"},"nullCount":{"id":0,"gender":0,"age":0,"interest":0}}


## User 1 changes interest: Sports → News

In [28]:
table = DeltaTable.forPath(spark, PATH)
table.update(condition="id = 1", set={'interest': "'News'"})

spark.read.format('delta').load(PATH).show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female| 30|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



## Parquet files can't be edited. So what happened on disk?

In [29]:
print('on disk :', files_on_disk())
print('in table:', files_in_table())

on disk : ['part-00000-2e7df052-8e80-4c27-8459-41dc02b5914c-c000.snappy.parquet', 'part-00000-4994dd5e-bec4-4ce9-b4fa-fe196ab77bd2-c000.snappy.parquet']
in table: ['part-00000-2e7df052-8e80-4c27-8459-41dc02b5914c-c000.snappy.parquet']


## Open version 1

In [30]:
v1 = commit(1)
print('remove:', v1.where('remove is not null').first().remove.path)
print('add   :', v1.where('add is not null').first().add.path)

remove: part-00000-4994dd5e-bec4-4ce9-b4fa-fe196ab77bd2-c000.snappy.parquet
add   : part-00000-2e7df052-8e80-4c27-8459-41dc02b5914c-c000.snappy.parquet


In [31]:
v1.printSchema()

root
 |-- add: struct (nullable = true)
 |    |-- dataChange: boolean (nullable = true)
 |    |-- modificationTime: long (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- size: long (nullable = true)
 |    |-- stats: string (nullable = true)
 |-- commitInfo: struct (nullable = true)
 |    |-- engineInfo: string (nullable = true)
 |    |-- isBlindAppend: boolean (nullable = true)
 |    |-- isolationLevel: string (nullable = true)
 |    |-- operation: string (nullable = true)
 |    |-- operationMetrics: struct (nullable = true)
 |    |    |-- executionTimeMs: string (nullable = true)
 |    |    |-- numAddedBytes: string (nullable = true)
 |    |    |-- numAddedChangeFiles: string (nullable = true)
 |    |    |-- numAddedFiles: string (nullable = true)
 |    |    |-- numCopiedRows: string (nullable = true)
 |    |    |-- numDeletionVectorsAdded: string (nullable = true)
 |    |    |-- numDeletionVectorsRemoved: string (nullable = true)
 |    |    |-- numDeletionVectorsU

In [12]:
metrics = v1.where('commitInfo is not null').first().commitInfo.operationMetrics
for name, value in sorted(metrics.asDict().items()):
    print(f'{name:28} {value}')

executionTimeMs              3063
numAddedBytes                1279
numAddedChangeFiles          0
numAddedFiles                1
numCopiedRows                3
numDeletionVectorsAdded      0
numDeletionVectorsRemoved    0
numDeletionVectorsUpdated    0
numRemovedBytes              1253
numRemovedFiles              1
numUpdatedRows               1
rewriteTimeMs                555
scanTimeMs                   2506


## A stray file lands in the folder

In [32]:
shutil.copy(f'{PATH}/{files_on_disk()[0]}', f'{PATH}/stray.parquet')

print('Parquet reader:', spark.read.parquet(PATH).count(), 'rows')
print('Delta reader  :', spark.read.format('delta').load(PATH).count(), 'rows')

Parquet reader: 12 rows
Delta reader  : 4 rows


In [14]:
os.remove(f'{PATH}/stray.parquet')

## User 3 leaves

In [33]:
table.delete("id = 3")

spark.read.format('delta').load(PATH).show()
print('on disk :', files_on_disk())
print('in table:', files_in_table())

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female| 30|    News|
|  4|Female| 28| Fashion|
+---+------+---+--------+

on disk : ['part-00000-2e7df052-8e80-4c27-8459-41dc02b5914c-c000.snappy.parquet', 'part-00000-4994dd5e-bec4-4ce9-b4fa-fe196ab77bd2-c000.snappy.parquet', 'part-00000-e5299094-3292-4e3d-8738-72fa0126c109-c000.snappy.parquet', 'stray.parquet']
in table: ['part-00000-e5299094-3292-4e3d-8738-72fa0126c109-c000.snappy.parquet']


## The table's history

In [16]:
table.history().select('version', 'timestamp', 'operation').show(truncate=False)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-11-10 04:36:40.305|DELETE   |
|1      |2026-11-10 04:36:23.875|UPDATE   |
|0      |2026-11-10 04:35:51.504|WRITE    |
+-------+-----------------------+---------+



**Old files are still on disk. Can we read the table as it was at version 0?** → next